In [ ]:
# ── Where a genome-scale model is incomplete ───────────────────────────────
#
# A GEM is not a finished description of metabolism, and it says so itself if
# you ask. This notebook asks, for Human-GEM: which metabolites have no route
# in or out, which reactions have no enzyme assigned, which parts of the
# network cannot carry flux, and where those gaps concentrate.
#
# The point is not to criticise the reconstruction — it is to produce a
# TARGET LIST. Every gap here is a place where a generative model can propose
# something without contradicting an existing annotation, which is the only
# setting in which a novel proposal is scoreable rather than automatically
# counted as an error.
#
# Everything is computed from the stoichiometry and the gene-reaction matrix,
# with sparse operations throughout — the dense S is 875 MB and nothing here
# needs it.
#
# Point GEM_DIR at another gem_ref/ to run the same analysis on a different
# reconstruction.

import os, sys, json, collections
from pathlib import Path
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

SETUP_DIR = None
def _find():
    if SETUP_DIR: return Path(SETUP_DIR).expanduser().resolve()
    own = globals().get("__vsc_ipynb_file__") or globals().get("__file__")
    cands = ([Path(own).resolve().parent] if own else [])
    here = Path.cwd().resolve()
    cands += [here, here / "data/gem"] + [p / "data/gem" for p in here.parents]
    cands += [Path("data/gem")]
    for c in cands:
        if (c / "torch_fba.py").exists(): return c.resolve()
    raise FileNotFoundError("could not find data/gem — set SETUP_DIR")

HERE = _find(); os.chdir(HERE); sys.path.insert(0, str(HERE))
GEM_DIR = HERE / "gem_ref"
FIG_DIR = HERE / "figures"; FIG_DIR.mkdir(exist_ok=True)
INK, SOFT, GRID = "#0b0b0b", "#52514e", "#e4e3de"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8880", "grid.color": GRID, "font.size": 9})
if not (GEM_DIR / "gem.npz").exists():
    raise FileNotFoundError(f"{GEM_DIR} missing — run build_gem.py first")

z = np.load(GEM_DIR / "gem.npz")
meta = json.loads((GEM_DIR / "meta.json").read_text())
n_mets, n_rxns = (int(x) for x in z["S_shape"])
S = sp.csr_matrix((z["S_val"], (z["S_row"], z["S_col"])), shape=(n_mets, n_rxns))
Sc = S.tocsc()
lb, ub = z["lb"], z["ub"]
rxns, mets, genes = meta["rxns"], meta["mets"], meta["genes"]
subs = np.array(meta["subsystems"])
RG = sp.csr_matrix((np.ones(len(z["rgm_row"])), (z["rgm_row"], z["rgm_col"])),
                   shape=(n_rxns, len(genes)))
print(f"{meta.get('version','?') or 'Human-GEM'}: {n_rxns:,} reactions, "
      f"{n_mets:,} metabolites, {len(genes):,} genes, {S.nnz:,} nonzeros")

# ── the four gap classes, computed once ────────────────────────────────────
#
# A metabolite can be PRODUCED if some reaction makes it in a direction that
# reaction is allowed to run; likewise CONSUMED. Anything failing either test
# cannot sit at steady state, so nothing through it can carry flux.
coo = S.tocoo()
fwd, bwd = ub > 0, lb < 0
makes = ((coo.data > 0) & fwd[coo.col]) | ((coo.data < 0) & bwd[coo.col])
takes = ((coo.data < 0) & fwd[coo.col]) | ((coo.data > 0) & bwd[coo.col])
prod = np.zeros(n_mets, bool); prod[coo.row[makes]] = True
cons = np.zeros(n_mets, bool); cons[coo.row[takes]] = True
present = np.diff(S.indptr) > 0
dead = present & ~(prod & cons)

orphan   = np.diff(RG.indptr) == 0                       # no gene rule
has_ec   = z["rxn_checkable"] | ~z["rxn_checkable"]      # placeholder, see cell 2
degree1  = present & (np.diff(S.indptr) == 1)
blocked  = np.asarray((S[dead] != 0).sum(0)).ravel() > 0  # touches a dead end

print(f"\nfour ways this model is incomplete:")
print(f"  metabolites with no route in or out   {int(dead.sum()):>6,}  "
      f"({dead.sum()/present.sum():.1%} of those used)")
print(f"  metabolites in exactly one reaction   {int(degree1.sum()):>6,}")
print(f"  reactions with no enzyme assigned     {int(orphan.sum()):>6,}  "
      f"({orphan.mean():.1%})")
print(f"  reactions that cannot carry flux      {int(blocked.sum()):>6,}  "
      f"({blocked.mean():.1%})")

In [ ]:
# ── 1. Dead ends: what kind of gap, and where ──────────────────────────────
#
# A dead end is not one thing. A metabolite the model can MAKE but never USE
# needs a consuming reaction — a sink. One it can use but never make needs a
# source. The two call for different proposals, so they are counted apart.
#
# The compartment breakdown is the actionable part. If the same metabolite
# exists in another compartment, the missing link is very likely a TRANSPORT
# reaction rather than new chemistry — a much narrower and more checkable
# claim, and one the curators are demonstrably still working through
# (57 of the 178 reactions added between v1.13.0 and v2.0.0 were transport).

need_sink   = present & prod & ~cons
need_source = present & cons & ~prod
comps = meta["comps"]
cname = dict(zip(meta["comps"], meta.get("comp_names", meta["comps"])))
mc = np.array(meta["met_comps"])
comp_of = np.array([comps[k-1] if 0 < k <= len(comps) else "?" for k in mc])

# would a transport reaction fix it? the same species in another compartment
core = np.array([m[:-1] for m in mets])
bycore = collections.defaultdict(list)
for i in np.flatnonzero(present):
    bycore[core[i]].append(i)
transportable = np.array([len(bycore[core[i]]) > 1 if dead[i] else False
                          for i in range(n_mets)])

print(f"{int(dead.sum()):,} dead-end metabolites")
print(f"  produced, never consumed  ->  needs a SINK    {int(need_sink.sum()):>5,}")
print(f"  consumed, never produced  ->  needs a SOURCE  {int(need_source.sum()):>5,}")
print(f"\n  the same species exists in another compartment: "
      f"{int(transportable.sum()):,} ({transportable.sum()/dead.sum():.0%})")
print(f"  -> a transport reaction would resolve those; the rest need chemistry")

# Each panel is its own figure, so any one can be dropped into a document at
# full width without the other two coming with it. The shared caption becomes
# a per-figure title.
CAP = "Dead-end metabolites — where Human-GEM cannot balance"

fig, ax = plt.subplots(figsize=(5.0, 3.6))
ax.bar([0, 1], [need_sink.sum(), need_source.sum()], color=["#c0392b", "#2a78d6"])
ax.set_xticks([0, 1])
ax.set_xticklabels(["needs a sink\n(made, never used)",
                    "needs a source\n(used, never made)"], fontsize=8)
for i, v in enumerate([need_sink.sum(), need_source.sum()]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9, color=SOFT)
ax.set_ylabel("metabolites")
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_deadends_a_kinds.png", dpi=200, bbox_inches="tight")
plt.show()

rows = []
for c in set(comp_of[present]):
    m = present & (comp_of == c)
    rows.append((cname.get(c, c), int((dead & m).sum()), int(m.sum())))
rows = sorted(rows, key=lambda r: -r[1] / max(r[2], 1))[:9]

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.barh(range(len(rows))[::-1], [r[1] / r[2] for r in rows], color="#0ea5a0")
ax.set_yticks(range(len(rows))[::-1])
ax.set_yticklabels([f"{r[0][:22]} ({r[1]}/{r[2]})" for r in rows], fontsize=8)
ax.axvline(dead.sum() / present.sum(), ls="--", lw=1, color=INK)
ax.set_xlabel("share of the compartment's metabolites that are dead ends")

ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_deadends_b_compartments.png", dpi=200,
            bbox_inches="tight")
plt.show()

n_t = int(transportable.sum()); n_o = int(dead.sum()) - n_t
fig, ax = plt.subplots(figsize=(4.6, 4.0))
ax.pie([n_t, n_o], labels=[f"transport would fix\n{n_t:,}",
                           f"needs new chemistry\n{n_o:,}"],
       colors=["#1baf7a", "#eda100"], autopct="%1.0f%%",
       textprops={"fontsize": 9}, startangle=90)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_deadends_c_fix.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ── 2. The catalysis gap: reactions with no enzyme ─────────────────────────
#
# 40% of Human-GEM's reactions have no gene rule. That is the largest single
# gap in the model and the most tractable one to propose into: there is no
# annotation to contradict, and for the subset that carries an EC number the
# chemistry is already characterised — only the protein is unknown.
#
# Those are the cleanest target this notebook produces. An EC number says what
# reaction class an enzyme must belong to, which is checkable against sequence
# and domain content independently of anything the GEM was built from.

import scipy.io as sio
from build_gem import as_str_list
mat = HERE / "gem_cache" / "Human-GEM.mat"
ec = None
if mat.exists():
    g_ = sio.loadmat(mat, struct_as_record=False, squeeze_me=True)["humanGEM"]
    ec = as_str_list(g_.eccodes)
    has_ec = np.array([bool(str(x).strip()) for x in ec])
else:
    has_ec = np.zeros(n_rxns, bool)
    print("Human-GEM.mat not cached — the EC breakdown is skipped")

npart = z["rxn_n_participants"]
print(f"{int(orphan.sum()):,} of {n_rxns:,} reactions have NO gene rule "
      f"({orphan.mean():.0%})")
if has_ec.any():
    tgt = orphan & has_ec & (npart > 1)
    print(f"  ...with an EC code (chemistry known, enzyme unknown)  "
          f"{int((orphan & has_ec).sum()):>5,}")
    print(f"  ...and not an exchange/demand pseudo-reaction         "
          f"{int(tgt.sum()):>5,}   <- the target list")
print(f"  reactions WITH a gene rule                             "
      f"{int((~orphan).sum()):>5,}   (proposing here contradicts an annotation)")

CAP = "The catalysis gap — 40% of reactions have no enzyme assigned"

cats = [("has a gene rule", int((~orphan).sum()), "#1baf7a"),
        ("orphan, EC known", int((orphan & has_ec).sum()), "#eda100"),
        ("orphan, no EC", int((orphan & ~has_ec).sum()), "#c0392b")]
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.bar(range(3), [c[1] for c in cats], color=[c[2] for c in cats])
ax.set_xticks(range(3)); ax.set_xticklabels([c[0] for c in cats], fontsize=8)
for i, c in enumerate(cats):
    ax.text(i, c[1], f"{c[1]:,}", ha="center", va="bottom", fontsize=9, color=SOFT)
ax.set_ylabel("reactions")
ax.set_title(f"{CAP}\nWho catalyses it?", loc="left", fontsize=10, color=INK)
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_orphans_a_who.png", dpi=200, bbox_inches="tight")
plt.show()

ng = np.diff(RG.indptr)
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.hist(ng[ng > 0], bins=np.arange(0.5, 22.5), color="#2a78d6")
ax.set_yscale("log"); ax.set_xlabel("genes assigned to the reaction")
ax.set_ylabel("reactions")
ax.set_title(f"{CAP}\nWhere a rule exists — median "
             f"{int(np.median(ng[ng>0]))} genes", loc="left", fontsize=10, color=INK)
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_orphans_b_rulesize.png", dpi=200, bbox_inches="tight")
plt.show()

rows = []
for s_ in set(subs):
    m = subs == s_
    if m.sum() >= 30:
        rows.append((s_, int(m.sum()), float(orphan[m].mean())))
rows = sorted(rows, key=lambda r: -r[2])[:10]
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.barh(range(len(rows))[::-1], [r[2] for r in rows], color="#c0392b")
ax.set_yticks(range(len(rows))[::-1])
ax.set_yticklabels([f"{r[0][:26]} ({r[1]})" for r in rows], fontsize=8)
ax.axvline(orphan.mean(), ls="--", lw=1, color=INK)
ax.set_xlabel("share of the subsystem's reactions with no gene")
ax.set_title(f"{CAP}\nWhere the enzymes are missing  (dashed = model-wide rate)",
             loc="left", fontsize=10, color=INK)
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_orphans_c_subsystems.png", dpi=200,
            bbox_inches="tight")
plt.show()


In [ ]:
# ── 3. What cannot carry flux, and what is disconnected ────────────────────
#
# A reaction touching a dead-end metabolite cannot hold a steady state, so it
# is blocked no matter what the bounds say. That is a lower bound on the truly
# blocked set — a full FVA would find more — but it is exact, needs no solver,
# and localises the damage.
#
# Connectivity is the other structural check. A metabolite in a small isolated
# component is not reachable from the rest of metabolism, which usually means
# a curation fragment rather than biology.

import scipy.sparse.csgraph as cg
print(f"{int(blocked.sum()):,} reactions touch at least one dead end "
      f"({blocked.mean():.1%})")
print(f"  they involve {int((S[:, blocked] != 0).sum()):,} metabolite "
      f"participations\n")

idx = np.flatnonzero(present)
A = (S[idx] != 0).astype(np.int8)
M = (A @ A.T).tocsr(); M.setdiag(0); M.eliminate_zeros()
ncomp, lab = cg.connected_components(M, directed=False)
sz = np.bincount(lab)
print(f"metabolite connectivity: {ncomp:,} components over {len(idx):,} metabolites")
print(f"  largest {sz.max():,} ({sz.max()/len(idx):.1%})   singletons {(sz==1).sum():,}"
      f"   components < 5 {(sz<5).sum():,}")

CAP = "Blocked and disconnected — the structural gaps"

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.hist(np.clip(sz, 1, 50), bins=np.arange(0.5, 51.5), color="#4a3aa7")
ax.set_yscale("log"); ax.set_xlabel("component size (clipped at 50)")
ax.set_ylabel("components")
ax.set_title(f"{CAP}\n{ncomp} components — all but one are fragments",
             loc="left", fontsize=10, color=INK)
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_blocked_a_components.png", dpi=200,
            bbox_inches="tight")
plt.show()

deg = np.diff(S.indptr)[present]
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.hist(np.clip(deg, 1, 40), bins=np.arange(0.5, 41.5), color="#0ea5a0")
ax.set_yscale("log")
ax.set_xlabel("reactions per metabolite (clipped at 40)")
ax.set_ylabel("metabolites")
ax.set_title(f"{CAP}\nDegree — {int((deg==1).sum()):,} appear in exactly one "
             f"reaction", loc="left", fontsize=10, color=INK)
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_blocked_b_degree.png", dpi=200, bbox_inches="tight")
plt.show()

rows = []
for s_ in set(subs):
    m = subs == s_
    if m.sum() >= 30:
        rows.append((s_, int(m.sum()), float(blocked[m].mean())))
rows = sorted(rows, key=lambda r: -r[2])[:10]
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.barh(range(len(rows))[::-1], [r[2] for r in rows], color="#eda100")
ax.set_yticks(range(len(rows))[::-1])
ax.set_yticklabels([f"{r[0][:26]} ({r[1]})" for r in rows], fontsize=8)
ax.set_xlabel("share of the subsystem's reactions that are blocked")
ax.set_title(f"{CAP}\nWhere flux stops", loc="left", fontsize=10, color=INK)
ax.grid(alpha=.3); ax.set_axisbelow(True); ax.tick_params(labelsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "gem_gaps_blocked_c_subsystems.png", dpi=200,
            bbox_inches="tight")
plt.show()


In [ ]:
# ── 4. The target list ─────────────────────────────────────────────────────
#
# Two CSVs, both ranked so the most tractable proposals come first.
#
# `gem_gaps_metabolites.csv` — every dead end, with the direction of the gap
# and whether the same species exists elsewhere (i.e. whether a transport
# reaction would close it).
#
# `gem_gaps_reactions.csv` — every orphan reaction, EC-annotated ones first,
# since those tell a model what protein family to look for and are checkable
# against sequence independently of the GEM.
#
# These are the denominators for a discovery claim. "We proposed enzymes for
# N of 408 EC-annotated orphans, M validated against held-out evidence" is a
# result whether M is large or small; "our graphs look plausible" is not.

import csv as _csv
mnames = meta.get("met_names", mets)
rnames = meta.get("rxn_names", rxns)

out_m = HERE / "gem_gaps_metabolites.csv"
with open(out_m, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["metabolite", "name", "compartment", "gap", "transport_would_fix",
                "n_reactions", "subsystems"])
    for i in np.flatnonzero(dead):
        rs = S.indices[S.indptr[i]:S.indptr[i+1]]
        w.writerow([mets[i], str(mnames[i])[:60], cname.get(comp_of[i], comp_of[i]),
                    "needs_sink" if need_sink[i] else "needs_source",
                    int(transportable[i]), len(rs),
                    ";".join(sorted({subs[j] for j in rs}))[:80]])

out_r = HERE / "gem_gaps_reactions.csv"
rank = np.lexsort((~(npart > 1), ~has_ec))          # EC first, then real reactions
with open(out_r, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["reaction", "name", "subsystem", "ec", "n_participants",
                "touches_dead_end"])
    for j in rank:
        if not orphan[j]:
            continue
        w.writerow([rxns[j], str(rnames[j])[:60], subs[j],
                    (str(ec[j]) if ec is not None else ""), int(npart[j]),
                    int(blocked[j])])

n_ec = int((orphan & has_ec & (npart > 1)).sum())
print(f"wrote {out_m.name}  ({int(dead.sum()):,} dead-end metabolites)")
print(f"      {out_r.name}  ({int(orphan.sum()):,} orphan reactions, "
      f"{n_ec:,} with an EC code)\n")
print("the shortlist — orphan reactions whose chemistry is already known:")
shown = 0
for j in rank:
    if not (orphan[j] and has_ec[j] and npart[j] > 1):
        continue
    print(f"  {rxns[j]:<12} EC {str(ec[j])[:18]:<20} {str(subs[j])[:28]:<30}"
          f"{str(rnames[j])[:30]}")
    shown += 1
    if shown >= 12:
        break
print(f"\n  {n_ec:,} in total — the cleanest place to point a generative model.")